# loss-item-scalar-extract — worked example 1: Verify .item() Returns a Python Float, Not a Tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `loss-item-scalar-extract`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Calling `.item()` on a zero-dimensional PyTorch tensor extracts its value as a plain Python scalar — a `float`, `int`, or `bool` depending on the tensor's dtype. This is essential for logging because logging frameworks like wandb and standard f-strings cannot serialize PyTorch tensors directly. The key requirement is that the input tensor must be zero-dimensional (a single scalar value); calling `.item()` on a multi-element tensor raises a `RuntimeError`.

## Worked solution

**Step 1 — create a 0-D loss tensor with a grad_fn.**
We build `loss = (w * x + b).pow(2)` where `w` and `x` are scalars with `requires_grad=True`. The result is a 0-D tensor whose `.grad_fn` is not None — exactly the kind of tensor produced by a real forward pass.

**Step 2 — call `.item()`.**
We call `loss.item()`, which synchronizes with the device, copies the single scalar value into Python memory, and returns a plain Python `float`. The call severs any connection to the autograd graph: the returned `float` has no gradient tracking at all.

**Step 3 — confirm the Python type.**
We check `type(scalar).__name__` and confirm it equals `'float'`. This is NOT a `torch.Tensor`, not a `numpy.float32` — it is a standard Python `float` that can be passed directly to `print`, `json.dumps`, or `wandb.log`.

**Step 4 — confirm the original tensor is unchanged.**
The `.item()` call does not mutate `loss`. The original tensor still has its `.grad_fn` attached, and we can still call `.backward()` on it if needed. The scalar extraction is a read-only operation.

In [ ]:
import torch as t

t.manual_seed(99)
w = t.tensor(2.5, requires_grad=True)
x = t.tensor(1.3, requires_grad=False)
b = t.tensor(-0.7, requires_grad=True)

# Forward: 0-D loss tensor with a real grad_fn
loss = (w * x + b).pow(2)

# Extract the scalar for logging
scalar = loss.item()

# Verify type and value
print(f"loss tensor: {loss}")
print(f"loss.item() -> {scalar}  type={type(scalar).__name__}")
print(f"Is Python float: {isinstance(scalar, float)}")
print(f"Original loss.grad_fn still intact: {loss.grad_fn is not None}")

# Confirm .item() agrees with direct float conversion
assert isinstance(scalar, float)
assert abs(scalar - float(loss.detach())) < 1e-6
print("All checks passed.")